# GTSRB Research: Experiment 3 — Frequency-Aware ResNet-18 Robustness Benchmark

This notebook implements **Experiment 3 (Spectral Data-Centric Robustness)** from our controlled research ladder:
$$X \to \text{Fourier Frequency Regularization (LFC / Spectral Dropout)} \to \text{ResNet-18} \to \text{43-Class Head}$$

### Methodological Grounding:
- **Wang, Wu, Yin, & Xing (CVPR 2020):** *High Frequency Component Helps Explain the Generalization of CNNs*.
- Directly addresses the **AugMix Noise Deficit** ($-5.96\%$ MCA regression) identified in Phase 1 diagnostics by mitigating reliance on high-frequency texture shortcuts without leaking test-set spatial corruptions.

### Experimental Controls (Strictly Identical to Baseline & AugMix):
1. **Architecture:** Pretrained ResNet-18 with Kaiming Normal 43-class head and Zero-$\gamma$ BN init.
2. **Optimization:** SGD (Nesterov $\mu=0.9$, lr=0.05, weight decay $10^{-4}$ with No-Bias-Decay rule).
3. **Schedule:** 3-epoch linear warmup + half-period Cosine Annealing decay over 25 epochs.
4. **Evaluation:** Clean accuracy, Macro-F1, ECE, and 15 held-out corruptions $\times$ 5 severities ($mCA$ and $RPD$).

In [ ]:
# --- Step 1: Environment Setup & GPU Verification ---
import os
from google.colab import drive
drive.mount('/content/drive')

# Check if dataset is already unzipped from prior runs
if not os.path.exists('/content/ML_Project/german'):
    print('Extracting base dataset from Drive...')
    !cp "/content/drive/MyDrive/ML_Project_Data.zip" /content/
    !unzip -q /content/ML_Project_Data.zip -d /content/ML_Project
else:
    print('Base dataset already present at /content/ML_Project!')

# Update Exp 3 code if uploaded as tiny code zip
if os.path.exists('/content/drive/MyDrive/exp3_frequency_code.zip'):
    print('Updating Exp 3 code from exp3_frequency_code.zip...')
    !unzip -o -q "/content/drive/MyDrive/exp3_frequency_code.zip" -d /content/ML_Project

%cd /content/ML_Project

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# --- Step 2: Frequency Augmentation Visual Inspection ---
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from src.dataset import GTSRBDataset, IMAGENET_MEAN, IMAGENET_STD
from src.augmentations.frequency import low_frequency_component, frequency_dropout

base_ds = GTSRBDataset('data/master_v1/metadata/train.csv', 'german', target_size=64, is_training=False)
print(f"Train samples: {len(base_ds):,}")

# Sample 4 distinct sign classes and visualize Clean vs LFC vs Frequency Dropout
fig, axes = plt.subplots(4, 3, figsize=(11, 12), dpi=120)
col_titles = ['Clean Canvas', 'Low-Frequency Component (LFC)', 'Frequency Dropout (Sev=3)']

for row_idx, sample_idx in enumerate([120, 650, 1800, 3100]):
    item = base_ds[sample_idx]
    # De-normalize tensor to PIL
    t = item['image'].cpu().numpy().transpose(1, 2, 0)
    img_np = (t * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
    pil_clean = Image.fromarray((img_np * 255).astype(np.uint8))

    pil_lfc = low_frequency_component(pil_clean, radius=14)
    pil_drop = frequency_dropout(pil_clean, severity=3.0, r_min=10)

    for col_idx, im_view in enumerate([pil_clean, pil_lfc, pil_drop]):
        ax = axes[row_idx, col_idx]
        ax.imshow(im_view)
        if row_idx == 0:
            ax.set_title(col_titles[col_idx], fontsize=11, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(item['class_name'], fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 3: Train Frequency-Aware ResNet-18 (Colab T4 GPU) ---
# 25 epochs takes ~6-8 minutes on T4 with Mixed Precision (AMP)
!python train_frequency.py \
    --data-root german \
    --meta-dir data/master_v1/metadata \
    --output-dir checkpoints/frequency_resnet18 \
    --seed 42 \
    --epochs 25 \
    --batch-size 128 \
    --lr 0.05 \
    --severity 3.0 \
    --dropout-prob 0.50 \
    --blend-prob 0.30

In [ ]:
# --- Step 4: Plot Loss & Validation Curves ---
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('checkpoints/frequency_resnet18/training_history.json', 'r') as f:
    history = json.load(f)

epochs_df = pd.DataFrame(history['epochs'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss Convergence
ax1.plot(epochs_df['epoch'], epochs_df['train_loss'], label='Train Loss (Freq Reg)', color='blue', lw=2)
ax1.plot(epochs_df['epoch'], epochs_df['val_loss'], label='Val Loss (Clean Canvas)', color='red', lw=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Frequency-Aware Training Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy & Macro-F1
ax2.plot(epochs_df['epoch'], epochs_df['val_acc'] * 100, label='Val Accuracy (%)', color='green', lw=2)
ax2.plot(epochs_df['epoch'], epochs_df['val_macro_f1'] * 100, label='Val Macro-F1 (%)', color='purple', lw=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score (%)')
ax2.set_title('Validation Performance')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 5: Clean Test & Robustness Benchmark (15 Corruptions x 5 Severities) ---
!python evaluate_robustness.py \
    --checkpoint checkpoints/frequency_resnet18/best_frequency_resnet18.pth \
    --data-root german \
    --meta-path data/master_v1/metadata/test.csv \
    --output-dir reports/frequency_resnet18 \
    --batch-size 128

In [ ]:
# --- Step 6: 3-Way Comparative Analysis: Baseline vs. AugMix vs. Frequency ---
import json
from pathlib import Path
import pandas as pd

p_base = Path('reports/baseline_resnet18/evaluation_summary.json')
p_aug = Path('reports/augmix_resnet18/evaluation_summary.json')
p_freq = Path('reports/frequency_resnet18/evaluation_summary.json')

rows = []
summaries = {}
for name, p in [('Baseline ResNet-18', p_base), ('AugMix ResNet-18', p_aug), ('Frequency ResNet-18', p_freq)]:
    if p.exists():
        with open(p) as f:
            summaries[name] = json.load(f)

if 'Frequency ResNet-18' in summaries:
    metrics = [
        ('Clean Accuracy', 'clean_accuracy', True),
        ('Clean Macro-F1', 'clean_macro_f1', True),
        ('Expected Calibration Error (ECE)', 'clean_ece', False),
        ('Mean Corruption Accuracy (mCA)', 'mean_corruption_accuracy', True),
        ('Relative Performance Drop (RPD)', 'relative_performance_drop', False),
    ]
    comp_records = []
    for title, key, is_pct in metrics:
        rec = {'Metric': title}
        for model_name, s in summaries.items():
            val = s.get(key, 0.0)
            rec[model_name] = f"{val*100:.2f}%" if is_pct else f"{val:.4f}"
        comp_records.append(rec)
    display(pd.DataFrame(comp_records))

    # Save backup to Google Drive
    !mkdir -p "/content/drive/MyDrive/ML_Project_Results/frequency_resnet18"
    !cp -r checkpoints/frequency_resnet18/* "/content/drive/MyDrive/ML_Project_Results/frequency_resnet18/"
    !cp -r reports/frequency_resnet18/* "/content/drive/MyDrive/ML_Project_Results/frequency_resnet18/"
    print("\nAll Frequency-Aware results backed up to Google Drive successfully!")
else:
    print("Frequency evaluation summary not found yet.")